In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
data_10X_dat = pd.read_csv('../10Xdatasets/all_donor_meta_all_peptides.csv', index_col=0)
data_10X_dat['new_index']  = range(data_10X_dat.shape[0])
data_10X_dat = data_10X_dat[data_10X_dat['donor'] != "donor3"]
data_10X_dat

,v_gene_TRA,v_gene_TRB,d_gene_TRA,d_gene_TRB,j_gene_TRA,j_gene_TRB,c_gene_TRA,c_gene_TRB,cdr3_TRA,cdr3_TRB,...,B0702_GPAESAAGL_NC,NR(B0801)_AAKGRGAAL_NC,antigen,peptide,orig_ix,n_counts,log_counts,n_genes,mt_fraction,new_index
barcode,,,,,,,,,,,,,,,,,,,,,
AGGGTGAGTATTACCG-18,TRAV19,TRBV20-1,NaN,TRBD2,TRAJ40,TRBJ2-3,TRAC,TRBC2,CALSEASGTYKYIF,CSAPSGEGRDTQYF,...,0.0,0.0,A0301_KLGGALQAK_IE-1_CMV_binder,KLGGALQAK,0,5478.0,3.738622,1830,0.058050,0
CTTGGCTTCGTTGCCT-25,TRAV23DV6,TRBV7-2,NaN,TRBD1,TRAJ48,TRBJ2-5,TRAC,TRBC2,CAAILFGNEKLTF,CASSLFDSQETQYF,...,0.0,0.0,unknown,NaN,1,4585.0,3.661339,1382,0.063250,1
ACGATACTCGCAGGCT-40,TRAV9-2,TRBV7-6,NaN,TRBD2,TRAJ49,TRBJ2-3,TRAC,TRBC2,CALSADTGNQFYF,CASSLFDSGRLDTQYF,...,0.0,0.0,unknown,NaN,2,3742.0,3.573104,1279,0.067611,2
ACGCCAGTCATGTCTT-8,TRAV12-3,TRBV20-1,NaN,TRBD1,TRAJ32,TRBJ2-7,TRAC,TRBC2,CAMNPAWGGATNKLIF,CSASPGDYEQYF,...,0.0,0.0,unknown,NaN,4,4368.0,3.640283,1669,0.054716,3
TTCTTAGCAAAGAATC-4,TRAV13-2,TRBV4-1,NaN,TRBD2,TRAJ37,TRBJ2-1,TRAC,TRBC2,CAETALGNTGKLIF,CASSHGKGGNEQFF,...,0.0,0.0,unknown,NaN,5,9129.0,3.960423,2683,0.044035,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GAAGCAGAGCAGGCTA-3,TRAV22,TRBV24-1,NaN,TRBD2,TRAJ47,TRBJ2-2,TRAC,TRBC2,CAVEPLYGNKLVF,CATSDRLAGGELFF,...,0.0,0.0,unknown,NaN,159412,1547.0,3.189490,789,0.135747,145474
CAGTCCTTCATCACCC-8,TRAV29DV5,TRBV7-6,NaN,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAATHASGYDKVIF,CASSYLAGDFTDTQYF,...,0.0,0.0,unknown,NaN,159414,3934.0,3.594834,1468,0.041688,145475
GACTACACACGGTAAG-3,TRAV21,TRBV6-6,NaN,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAVDLMKTSYDKVIF,CASRTGLASTDTQYF,...,0.0,0.0,A1101_IVTDFSVIK_EBNA-3B_EBV_binder,IVTDFSVIK,159415,4267.0,3.630123,1646,0.047574,145476


In [3]:
ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides_harmony.h5ad')

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [4]:
ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [5]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides_harmony.h5ad')
gene_TCR = gene_TCR[data_10X_dat['new_index'].values,:]
gene_TCR

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


View of AnnData object with n_obs × n_vars = 115245 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 

In [6]:
gene_TCR.obs

,v_gene_TRA,v_gene_TRB,d_gene_TRA,d_gene_TRB,j_gene_TRA,j_gene_TRB,c_gene_TRA,c_gene_TRB,cdr3_TRA,cdr3_TRB,...,A2402_AYSSAGASI_NC,B0702_GPAESAAGL_NC,NR(B0801)_AAKGRGAAL_NC,antigen,peptide,orig_ix,n_counts,log_counts,n_genes,mt_fraction
barcode,,,,,,,,,,,,,,,,,,,,,
AGGGTGAGTATTACCG-18,TRAV19,TRBV20-1,None,TRBD2,TRAJ40,TRBJ2-3,TRAC,TRBC2,CALSEASGTYKYIF,CSAPSGEGRDTQYF,...,0.0,0.0,0.0,A0301_KLGGALQAK_IE-1_CMV_binder,KLGGALQAK,0,5478.0,3.738622,1830,0.058050
CTTGGCTTCGTTGCCT-25,TRAV23DV6,TRBV7-2,None,TRBD1,TRAJ48,TRBJ2-5,TRAC,TRBC2,CAAILFGNEKLTF,CASSLFDSQETQYF,...,0.0,0.0,0.0,unknown,nan,1,4585.0,3.661339,1382,0.063250
ACGATACTCGCAGGCT-40,TRAV9-2,TRBV7-6,None,TRBD2,TRAJ49,TRBJ2-3,TRAC,TRBC2,CALSADTGNQFYF,CASSLFDSGRLDTQYF,...,0.0,0.0,0.0,unknown,nan,2,3742.0,3.573104,1279,0.067611
ACGCCAGTCATGTCTT-8,TRAV12-3,TRBV20-1,None,TRBD1,TRAJ32,TRBJ2-7,TRAC,TRBC2,CAMNPAWGGATNKLIF,CSASPGDYEQYF,...,0.0,0.0,0.0,unknown,nan,4,4368.0,3.640283,1669,0.054716
TTCTTAGCAAAGAATC-4,TRAV13-2,TRBV4-1,None,TRBD2,TRAJ37,TRBJ2-1,TRAC,TRBC2,CAETALGNTGKLIF,CASSHGKGGNEQFF,...,0.0,0.0,0.0,unknown,nan,5,9129.0,3.960423,2683,0.044035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GAAGCAGAGCAGGCTA-3,TRAV22,TRBV24-1,None,TRBD2,TRAJ47,TRBJ2-2,TRAC,TRBC2,CAVEPLYGNKLVF,CATSDRLAGGELFF,...,0.0,0.0,0.0,unknown,nan,159412,1547.0,3.189490,789,0.135747
CAGTCCTTCATCACCC-8,TRAV29DV5,TRBV7-6,None,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAATHASGYDKVIF,CASSYLAGDFTDTQYF,...,0.0,0.0,0.0,unknown,nan,159414,3934.0,3.594834,1468,0.041688
GACTACACACGGTAAG-3,TRAV21,TRBV6-6,None,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAVDLMKTSYDKVIF,CASRTGLASTDTQYF,...,0.0,0.0,0.0,A1101_IVTDFSVIK_EBNA-3B_EBV_binder,IVTDFSVIK,159415,4267.0,3.630123,1646,0.047574


In [7]:
gene = pd.DataFrame(gene_TCR.obsm['X_pca_harmony'])
gene

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,3.036558,0.470259,-1.534361,1.494185,-2.802884,0.413915,-2.406211,1.511487,2.227762,-0.900201,...,-1.173055,0.916518,1.002145,-0.716572,1.147374,1.177967,-0.394466,0.175063,0.101577,-0.650903
1,3.568292,-4.473117,1.128025,-1.154003,-0.340075,-0.331987,-0.711000,0.480116,1.402941,-1.596925,...,0.252037,-0.189764,-0.193532,0.481584,0.704797,-0.477740,-0.321027,-0.303495,-0.785064,-0.610597
2,1.598286,-6.589628,-0.093850,-1.273293,-0.144133,0.581436,1.557961,-0.735445,-2.621554,0.874783,...,-0.040988,-0.404409,-0.550357,0.698925,0.180182,0.012100,-0.074760,0.533318,0.216403,0.333611
3,1.801606,2.185353,-0.062215,0.000600,-0.906292,2.433735,-0.400967,0.401172,-0.236139,0.913400,...,-0.799589,1.570934,0.222576,-0.422355,0.685801,0.413194,-1.090396,0.127215,0.058785,0.415131
4,10.459624,1.670054,-5.228515,5.566212,-5.204177,0.865626,2.290748,-1.135377,1.517275,2.510904,...,-0.023412,1.371591,-0.778464,-0.684442,-0.394926,0.551291,-0.294668,0.061575,-0.203088,-1.092191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115240,-16.187646,1.508734,-1.991076,-0.155738,0.383084,-0.886498,1.670027,-1.473262,0.665272,-0.767261,...,0.104526,-0.036424,0.123254,-0.486086,-0.272758,-0.008538,-0.298798,-0.302226,-0.595866,0.084635
115241,1.053205,-1.751885,0.171986,-0.500772,0.890879,1.848117,-0.074121,-2.129117,0.646666,-1.529322,...,0.050863,-0.160187,-0.953540,1.035829,-0.226151,-0.638986,-0.062691,0.617543,-0.507099,-1.065667
115242,0.229644,3.210759,-5.414100,-1.201439,0.934827,-0.937627,-0.753186,-0.134533,-0.124110,0.155970,...,-0.065988,0.211532,-0.294399,0.923304,-0.298969,-0.186653,0.899904,-0.126854,0.541319,0.687603
115243,-16.304995,-2.528540,3.613680,0.796308,0.277799,-0.970278,1.526098,-0.509714,0.608519,-0.181751,...,-0.517029,0.294368,-1.016489,0.503258,-0.715753,-0.543770,-0.070787,0.583562,-0.147427,0.412472


In [8]:
tcr_seq = gene_TCR.obs[['cdr3_TRB']]
tcr_seq

,cdr3_TRB
barcode,
AGGGTGAGTATTACCG-18,CSAPSGEGRDTQYF
CTTGGCTTCGTTGCCT-25,CASSLFDSQETQYF
ACGATACTCGCAGGCT-40,CASSLFDSGRLDTQYF
ACGCCAGTCATGTCTT-8,CSASPGDYEQYF
TTCTTAGCAAAGAATC-4,CASSHGKGGNEQFF
...,...
GAAGCAGAGCAGGCTA-3,CATSDRLAGGELFF
CAGTCCTTCATCACCC-8,CASSYLAGDFTDTQYF
GACTACACACGGTAAG-3,CASRTGLASTDTQYF


In [9]:
import tensorflow as tf

2026-02-16 22:12:01.362621: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 22:12:01.482566: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 22:12:01.511187: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal
# Define input layer
input_gex = Input(shape=(50,))
gex = Dense(units=64, kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(8,8,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121)(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
              loss='mse')

# Check layer names
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 50)]              0         
                                                                 
 dense (Dense)               (None, 64)                3264      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d (Conv2D)             (None, 8, 8, 64)          640       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense_1 (Dense)             (None, 64)                131136

2026-02-16 22:12:02.794909: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 22:12:02.924906: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20999 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:17:00.0, compute capability: 8.6


In [11]:
# import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten
# from tensorflow.keras.initializers import HeNormal

# # Define input layer
# input_gex = Input(shape=(100,))
# gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
# gex = Reshape(target_shape=(8,8,1))(gex)

# # Convolutional layers
# gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
# gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

# gex = Flatten()(gex)
# hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# # Transposed Convolutional layers
# tcr = Reshape(target_shape=(8,8,1))(hidden_layer)
# tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
# tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)

# tcr = Flatten()(tcr)
# tcr = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(tcr)
# tcr = Dense(units=121)(tcr)  # Ensure proper activation

# # Define model
# model = Model(inputs=input_gex, outputs=tcr)
# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
#               loss='mse')

# # Check layer names
# model.summary()


In [12]:
AE_tcr = pd.read_csv("../AE_emb_TRB_all_peptides_10X_all_donors.csv")
AE_tcr = AE_tcr.iloc[data_10X_dat['new_index'].values]
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,-1541.692100,431.963380,-113.48751,363.83960,-617.465500,-576.38770,-152.861240,-985.72640,445.512240,-236.100420,...,28.279720,1018.37134,261.223200,87.342865,-228.999980,146.428400,532.794070,865.979250,-578.558960,855.616900
1,-273.513100,-398.893860,616.80430,-877.82350,-871.226700,375.48306,422.579040,192.50789,-199.132490,-265.096560,...,494.893800,258.40955,-404.777220,-428.136350,-191.204730,-372.523960,203.390100,1186.992800,96.218480,-457.060200
2,380.871000,58.110577,-132.31460,-259.36774,-274.578430,-194.46793,248.315840,265.90674,-60.173637,-98.568726,...,92.126270,29.88193,-891.704800,370.281830,59.372814,-729.678000,902.504940,1161.101400,-700.613340,-425.839780
3,-1229.993200,261.055080,-116.35792,339.64825,-440.477600,65.39269,-221.381640,-939.52580,145.774370,39.421880,...,153.995830,770.41030,-417.146600,194.590100,-287.056100,-4.876206,-74.260704,43.650208,-67.906160,-309.779940
4,-1113.337300,-93.408485,-280.13140,-150.60530,-93.476110,-520.02313,64.995720,-1008.05770,174.263440,-62.470676,...,-56.364480,984.19050,467.822940,247.904310,409.301880,-60.920048,744.377300,1218.065600,-456.735320,617.486100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,-512.594500,-576.577700,-841.85020,-472.11673,-47.690334,-245.14507,299.702820,-409.62906,-274.262500,191.512990,...,-605.121150,1250.64900,-388.322940,282.790900,-704.643740,37.460022,647.884160,1549.309400,352.393280,-108.645310
145475,226.274380,956.163450,176.05064,-689.92920,162.394490,608.82790,-269.477480,-744.45300,-219.842730,835.156560,...,31.987225,508.31207,-341.937840,-267.425480,-786.679300,-816.931300,291.385250,466.537630,-74.314735,-342.459630
145476,1080.433300,373.862700,-65.57249,-259.77700,35.480648,-554.94806,-441.794400,93.13507,-163.760620,155.478730,...,-251.648400,687.42706,-52.241924,-690.953600,523.990230,-314.377440,-316.330050,697.732100,40.809692,-1766.797400
145477,55.882103,-766.636600,229.29408,-177.78737,634.726900,-295.03433,92.827480,443.62762,244.667330,-787.343600,...,385.697330,895.55630,-700.939200,-126.450980,-803.600100,133.600450,-130.163740,348.644740,-400.828250,38.264355


In [13]:
batch_gene = pd.read_csv

In [14]:
# import numpy as np
# from sklearn.decomposition import NMF

# # Generate random non-negative data
# data = gene.to_numpy()

# # Initialize the NMF model
# n_components = 100
# model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# # Fit the model to the data
# W = model_nmf.fit_transform(data)
# H = model_nmf.components_

# # Display the results
# print("Basis matrix (W):\n", W)
# print("Coefficients matrix (H):\n", H)


In [15]:

# W = pd.DataFrame(W)
# W

In [16]:
# # W.to_csv('gex_nmf_100_components_all_peptides_10X_donors_124.csv', index=False)
# W = pd.read_csv('gex_nmf_100_components_all_peptides_10X_donors_124.csv')
# W

In [17]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5, monitor='loss', min_delta=100)

history = model.fit(gene,AE_tcr, 
                epochs=1400, 
                batch_size=128, 
                shuffle = True,
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                callbacks=[reduce_learning_rate]
                )

Epoch 1/1400


2026-02-16 22:12:06.520039: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-16 22:12:07.063575: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2026-02-16 22:12:07.749214: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


901/901 [==============================] - 5s 3ms/step - loss: 217333.9844 - lr: 0.0015
Epoch 2/1400
901/901 [==============================] - 3s 3ms/step - loss: 187079.0156 - lr: 0.0015
Epoch 3/1400
901/901 [==============================] - 3s 3ms/step - loss: 175703.7188 - lr: 0.0015
Epoch 4/1400
901/901 [==============================] - 3s 3ms/step - loss: 169327.4375 - lr: 0.0015
Epoch 5/1400
901/901 [==============================] - 3s 3ms/step - loss: 164744.4062 - lr: 0.0015
Epoch 6/1400
901/901 [==============================] - 3s 3ms/step - loss: 161872.1094 - lr: 0.0015
Epoch 7/1400
901/901 [==============================] - 3s 3ms/step - loss: 159715.7812 - lr: 0.0015
Epoch 8/1400
901/901 [==============================] - 3s 3ms/step - loss: 157941.2031 - lr: 0.0015
Epoch 9/1400
901/901 [==============================] - 3s 4ms/step - loss: 156421.5156 - lr: 0.0015
Epoch 10/1400
901/901 [==============================] - 3s 3ms/step - loss: 155087.9219 - lr: 0.0015
Ep

In [18]:
for layer in model.layers:
    print(layer.name)

input_1
dense
reshape
conv2d
conv2d_1
flatten
dense_1
reshape_1
conv2d_transpose
conv2d_transpose_1
dropout
flatten_1
dense_2
dense_3
dense_4


In [19]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [21]:
# model.predict( W.iloc[1:2])

In [ ]:
# model.predict( W.iloc[4:5])

In [22]:
integration_pred = latent_model.predict( gene)

3602/3602 [==============================] - 3s 898us/step


In [23]:
pd.DataFrame(integration_pred[1:50,1:50])

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,369.354797,0.0,0.0,0.0,0.0,314.050598,0.0,136.243042,0.0,88.881935,...,0.0,0.0,52.157631,0.0,0.000000,160.374054,268.080811,351.611908,0.0,0.000000
1,267.162903,0.0,0.0,0.0,0.0,185.286926,0.0,26.296875,0.0,224.131378,...,0.0,0.0,238.227661,0.0,0.000000,51.959568,256.096313,453.779083,0.0,0.000000
2,129.664169,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,...,0.0,0.0,12.950380,0.0,0.000000,132.596451,226.383698,454.165131,0.0,21.886717
3,72.613419,0.0,0.0,0.0,0.0,234.225327,0.0,0.000000,0.0,193.881775,...,0.0,0.0,245.024246,0.0,0.000000,216.846954,274.632660,60.049553,0.0,0.000000
4,290.195953,0.0,0.0,0.0,0.0,359.714355,0.0,20.739758,0.0,0.000000,...,0.0,0.0,222.418457,0.0,0.000000,343.470795,202.409119,229.761826,0.0,140.110016
5,138.576523,0.0,0.0,0.0,0.0,259.986877,0.0,0.000000,0.0,260.673889,...,0.0,0.0,24.487440,0.0,0.000000,371.322510,224.917633,399.844177,0.0,374.042908
6,246.132217,0.0,0.0,0.0,0.0,399.687958,0.0,0.000000,0.0,57.546829,...,0.0,0.0,301.584930,0.0,0.000000,183.760376,134.913940,392.319092,0.0,11.687193
7,500.992157,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,154.164658,...,0.0,0.0,148.105606,0.0,351.770416,226.167099,226.141449,0.000000,0.0,0.000000
8,281.841614,0.0,0.0,0.0,0.0,105.339622,0.0,33.378788,0.0,162.913940,...,0.0,0.0,63.097946,0.0,0.000000,303.259644,285.537476,0.000000,0.0,107.686073
9,66.979309,0.0,0.0,0.0,0.0,87.830215,0.0,232.429062,0.0,40.805786,...,0.0,0.0,64.110420,0.0,0.000000,139.238052,283.310730,235.808426,0.0,76.756660


In [24]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [25]:
integration_pred.shape

(115245, 64)

In [26]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(115245, 64)

In [27]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,0.0,9.992558,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,176.207199,0.0,7.142615,211.374924,192.632462,0.0,0.000000
1,0.0,369.354797,0.0,0.0,0.0,0.0,314.050598,0.0,136.243042,0.0,...,0.0,0.0,0.0,0.000000,0.0,26.791628,237.986389,128.433548,0.0,92.402115
2,0.0,267.162903,0.0,0.0,0.0,0.0,185.286926,0.0,26.296875,0.0,...,0.0,0.0,0.0,120.880852,0.0,0.000000,232.813080,234.062683,0.0,230.338715
3,0.0,129.664169,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.000000,329.478973,201.747818,0.0,190.400681
4,0.0,72.613419,0.0,0.0,0.0,0.0,234.225327,0.0,0.000000,0.0,...,0.0,0.0,0.0,145.270996,0.0,16.029591,212.999481,79.993126,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115240,0.0,86.302139,0.0,0.0,0.0,0.0,429.631378,0.0,104.073906,0.0,...,0.0,0.0,0.0,314.169281,0.0,0.000000,291.204193,181.304077,0.0,55.415058
115241,0.0,138.715454,0.0,0.0,0.0,0.0,303.363312,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.0,33.402714,265.256226,280.637787,0.0,308.726044
115242,0.0,304.460144,0.0,0.0,0.0,0.0,298.014496,0.0,44.450897,0.0,...,0.0,0.0,0.0,0.000000,0.0,18.947437,236.268845,0.000000,0.0,183.998840
115243,0.0,207.695267,0.0,0.0,0.0,0.0,66.892937,0.0,41.138000,0.0,...,0.0,0.0,0.0,99.769478,0.0,268.095978,288.527283,433.653198,0.0,0.000000


In [28]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_beta_chain_10X_donor_124_peptides_batch_gene_64.csv", index=False)